## Imports

In [ ]:
%pip install pandas
%pip install scikit-learn
%pip install tensorflow
%pip install matplotlib
%pip install imbalanced-learn
%env SCIPY_ARRAY_API=1
%pip install xgboost lightgbm numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached scikit_learn-1.7.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scikit_learn-1.7.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.16.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached scipy-1.16.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached joblib-1.5.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached joblib-1.5.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.1-cp313-cp313-macosx_12_0_arm64.whl (8.6 MB)
Using cached joblib-1.5.1-py3-none-any.whl (307 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.1-cp313-cp313-macosx_12_0_arm64.whl (8.6 MB)
Using cached joblib-1.5.1-py3-none-any.whl (307 kB)
   ━━━━━━━━━━━━━━━━━━━

In [2]:
# 📦 Imports
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.data_utils import load_and_preprocess_data
from src.utils.data_utils import preprocess_input
from src.model import build_weighted_model  # make sure this is imported
import csv
import pandas as pd
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
import joblib


In [3]:
# --- 1. Load the trained model and scaler
model = load_model("../models/cnn_rnn_model.h5")
scaler = joblib.load("../models/scaler.pkl")

2025-07-01 12:58:40.585485: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-07-01 12:58:40.585702: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-07-01 12:58:40.585727: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-07-01 12:58:40.585785: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-01 12:58:40.585805: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
# --- 2. Load the full sensor dataset
df = pd.read_csv("../data/robot_sensor_data.csv")  # adjust path as needed

In [5]:
# --- 3. Preprocess all rows for model input
X_all, _ = preprocess_input(df, scaler)

/Users/LucaZagaia/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [6]:
# --- 4. Predict probabilities for each row
probs = model.predict(X_all).flatten()
labels = (probs > 0.5).astype(int)  # You can adjust the threshold if desired

24/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 

2025-07-01 12:58:42.151835: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [ ]:
# --- 5. Attach predictions to the dataframe
df['predicted_prob'] = probs
df['predicted_label'] = labels

In [8]:
# --- 6. Save predictions to a new CSV file
df.to_csv("../results/predictions.csv", index=False)
print("✅ Batch predictions saved to ../results/predictions.csv")

✅ Batch predictions saved to ../results/predictions.csv


In [1]:
import pandas as pd

df_pred = pd.read_csv("../results/predictions.csv")
print(df_pred['predicted_label'].value_counts())   # 0 = OK, 1 = FAILURE

print(df_pred['predicted_prob'].describe())


predicted_label
0    2000
Name: count, dtype: int64
count    2.000000e+03
mean     1.056884e-02
std      3.470315e-18
min      1.056884e-02
25%      1.056884e-02
50%      1.056884e-02
75%      1.056884e-02
max      1.056884e-02
Name: predicted_prob, dtype: float64
